In [72]:
from langgraph.graph import StateGraph,START,END
from typing import Annotated
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver
import os
load_dotenv()

True

In [30]:
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class ChatState(BaseModel):
    user_input: Annotated[list[BaseMessage], add_messages]
    bot_response: Annotated[list[BaseMessage], add_messages]

In [48]:
model = ChatGroq(
    model_name="openai/gpt-oss-120b",
    api_key=os.environ.get("GROQ_API_KEY"),
    temperature=0.7,
    max_tokens=150,
    stop=None,
)

In [49]:
graph = StateGraph(ChatState)

In [74]:
def ChatGroq_llm(state: ChatState):
    conversation = []
    for index, user_message in enumerate(state.user_input):
        conversation.append(f"User: {user_message.content}")
        if index < len(state.bot_response):
            conversation.append(f"Assistant: {state.bot_response[index].content}")

    prompt = "You are a helpful assistant. Use the conversation history to answer the latest question.\n\n"
    prompt += "\n".join(conversation)
    bot_response = model.invoke(input=prompt)
    return {"bot_response": [bot_response]}

In [75]:
# Recreate the graph so this cell can be safely rerun
checkpoint = MemorySaver()
graph = StateGraph(ChatState)
graph.add_node("LLm_node", ChatGroq_llm)
graph.add_edge(START, "LLm_node")
graph.add_edge("LLm_node", END)
chatbot = graph.compile(checkpointer=checkpoint)

In [77]:
from langchain_core.messages import HumanMessage

thread_id = "demo-chat"
config = {"configurable": {"thread_id": thread_id}}

state = chatbot.invoke(
    {
        "user_input": [HumanMessage(content="My name is Pritam.")],
        "bot_response": [],
    },
    config=config,
)
print(f"Response: {state['bot_response'][-1].content}")

state = chatbot.invoke(
    {
        "user_input": [HumanMessage(content="What is my name?")],
        "bot_response": [],
    },
    config=config,
)
print(f"Response: {state['bot_response'][-1].content}")

Response: Got it—your name is Pritam. How can I help you today?
Response: Your name is Pritam.


In [78]:
thread_id = "interactive-chat"
config = {"configurable": {"thread_id": thread_id}}

while True:
    user_input = input("\nYou: ")
    if user_input.lower() in ["exit", "quit"]:
        break

    state = chatbot.invoke(
        {
            "user_input": [HumanMessage(content=user_input)],
            "bot_response": [],
        },
        config=config,
    )

    print(f"Response: {state['bot_response'][-1].content}")

Response: Hello, Pritam! Nice to meet you. How can I help you today?
Response: Your name is **Pritam Tung**.
Response: 100 + 10 = **110**.
Response: Sure!  

\( (100 + 10) \times 2 = 110 \times 2 = \mathbf{220} \)
Response: **FastAPI** is a modern, high‑performance web framework for building APIs with Python 3.7+.

### Key points

| Feature | Why it matters |
|---------|----------------|
| **Fast** | Built on **Starlette** (for the web parts) and **Pydantic** (for data validation). Under the hood it uses **uvicorn** and **ASGI**, giving performance comparable to Node.js or Go. |
| **Type‑hints first** | You
Response: Good night, Pritam! 🌙 I hope you have a restful sleep. If you need anything else tomorrow, just let me know. Take care!
Response: I’m actually a product of OpenAI, so I’m part of the same team that builds the models you’re thinking of. If you’ve noticed something that works better elsewhere, I’d love to hear more details so I can help you more effectively. Is there a
Respo